# NB0 — Préparation des données M5 (CA_1)

**Objectif :** Découper les fichiers sources en fichiers pré-découpés par période,
de façon à ce que NB1 (`forecasting_M5_antileakage.ipynb`) puisse reconstituer
exactement les mêmes `train_df` et `prices_df` qu'il aurait avec les fichiers bruts.

## Fichiers générés

### Dans `C:\Users\ayady\Downloads\train\`
| Fichier | Contenu |
|---------|--------|
| `historique_sales_CA1.csv` | Ventes CA_1, format wide, colonnes d_1 → d_1885 |
| `historique_prices_CA1.csv` | Prix CA_1, semaines wm_yr_wk couvrant jusqu'à d_1885 (wk ≤ 11608) |
| `calendar.csv` | Calendrier complet (copie) |

### Dans `C:\Users\ayady\Downloads\test\`
| Fichier | Contenu |
|---------|--------|
| `sales_du_mois_CA1.csv` | Ventes CA_1, colonnes d_1886 → d_1913 |
| `verite_terrain_CA1.csv` | Ventes CA_1, colonnes d_1914 → d_1941 (jamais vues en train) |
| `prices_du_mois_CA1.csv` | Prix CA_1, semaines wm_yr_wk couvrant d_1886 → d_1941 (wk 11609-11621) |

## Comment NB1 reconstitue train_df et prices_df

```python
# train_df = historique (wide d_1-1885) + mois (wide d_1886-1913) fusionnés en wide d_1-1913
# puis filtré sur CA_1 (déjà fait) → identique au train_df original filtré store_id=CA_1

# prices_df = historique_prices + prices_du_mois empilés
# → identique au prices_df original filtré store_id=CA_1
```

In [1]:
import pandas as pd
import numpy as np
import os, gc, shutil, time

# ── Chemins sources (fichiers bruts originaux) ────────────────────────────────
RAW_DIR        = r"C:\Users\ayady\OneDrive\Desktop\pfe\data\raw_data"
PATH_SALES     = os.path.join(RAW_DIR, "sales_train_evaluation.csv")
PATH_PRICES    = os.path.join(RAW_DIR, "sell_prices.csv")
PATH_CALENDAR  = os.path.join(RAW_DIR, "calendar.csv")

# ── Chemins de sortie ─────────────────────────────────────────────────────────
TRAIN_DIR = r"C:\Users\ayady\Downloads\train"
TEST_DIR  = r"C:\Users\ayady\Downloads\test"
os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(TEST_DIR,  exist_ok=True)

# ── Paramètres ────────────────────────────────────────────────────────────────
STORE_ID   = "CA_1"

# Frontières jours
HIST_END   = 1885   # d_1    → d_1885  : historique
MOIS_END   = 1913   # d_1886 → d_1913  : mois de validation
VERITE_END = 1941   # d_1914 → d_1941  : vérité terrain (jamais vue en train)

# Frontière semaines calendaires (wm_yr_wk)
# d_1885 = wk 11609 (samedi) → on coupe les prix à wk 11608 inclus
# pour garantir qu'aucune info de d_1886+ ne se glisse dans les stats historiques
WK_HIST_MAX  = 11608   # dernière semaine avant d_1886
WK_FUTUR_MIN = 11609   # première semaine couvrant d_1886 et au-delà

print("Configuration :")
print(f"  Store       : {STORE_ID}")
print(f"  Historique  : d_1 → d_{HIST_END}")
print(f"  Mois valid  : d_{HIST_END+1} → d_{MOIS_END}")
print(f"  Vérité      : d_{MOIS_END+1} → d_{VERITE_END}")
print(f"  Prix hist   : wk ≤ {WK_HIST_MAX}")
print(f"  Prix futur  : wk ≥ {WK_FUTUR_MIN}")

Configuration :
  Store       : CA_1
  Historique  : d_1 → d_1885
  Mois valid  : d_1886 → d_1913
  Vérité      : d_1914 → d_1941
  Prix hist   : wk ≤ 11608
  Prix futur  : wk ≥ 11609


## Étape 1 — Lecture et filtrage CA_1

In [2]:
print("Lecture sales_train_evaluation.csv ...")
t0 = time.time()
sales_raw = pd.read_csv(PATH_SALES)
print(f"  Shape brut : {sales_raw.shape}  ({time.time()-t0:.1f}s)")

# Filtrer CA_1 — identique à ce que fait NB1 cell 7
ca1 = sales_raw[sales_raw["store_id"] == STORE_ID].reset_index(drop=True)
del sales_raw; gc.collect()

print(f"  CA_1 : {len(ca1)} produits × {len(ca1.columns)} colonnes")

# Colonnes méta et colonnes jours disponibles
META_COLS = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
day_cols_all = [c for c in ca1.columns if c.startswith("d_")]
print(f"  Jours disponibles : {day_cols_all[0]} → {day_cols_all[-1]}  ({len(day_cols_all)} jours)")

Lecture sales_train_evaluation.csv ...
  Shape brut : (30490, 1947)  (2.1s)
  CA_1 : 3049 produits × 1947 colonnes
  Jours disponibles : d_1 → d_1941  (1941 jours)


## Étape 2 — Découpage des ventes en 3 fichiers

In [3]:
# ── Colonnes par période ───────────────────────────────────────────────────────
hist_day_cols   = [f"d_{i}" for i in range(1,          HIST_END  + 1) if f"d_{i}" in ca1.columns]
mois_day_cols   = [f"d_{i}" for i in range(HIST_END+1, MOIS_END  + 1) if f"d_{i}" in ca1.columns]
verite_day_cols = [f"d_{i}" for i in range(MOIS_END+1, VERITE_END+ 1) if f"d_{i}" in ca1.columns]

print(f"Jours historique   : {len(hist_day_cols)}  ({hist_day_cols[0]} → {hist_day_cols[-1]})")
print(f"Jours mois valid   : {len(mois_day_cols)}  ({mois_day_cols[0]} → {mois_day_cols[-1]})")
print(f"Jours vérité       : {len(verite_day_cols)}  ({verite_day_cols[0]} → {verite_day_cols[-1]})")

# ── 1. historique_sales_CA1.csv  (d_1 → d_1885) ──────────────────────────────
path_hist_sales = os.path.join(TRAIN_DIR, "historique_sales_CA1.csv")
ca1[META_COLS + hist_day_cols].to_csv(path_hist_sales, index=False)
print(f"\n✓ historique_sales_CA1.csv     → {path_hist_sales}")
print(f"  Shape : {ca1[META_COLS + hist_day_cols].shape}")

# ── 2. sales_du_mois_CA1.csv     (d_1886 → d_1913) ───────────────────────────
path_mois_sales = os.path.join(TEST_DIR, "sales_du_mois_CA1.csv")
ca1[META_COLS + mois_day_cols].to_csv(path_mois_sales, index=False)
print(f"✓ sales_du_mois_CA1.csv        → {path_mois_sales}")
print(f"  Shape : {ca1[META_COLS + mois_day_cols].shape}")

# ── 3. verite_terrain_CA1.csv    (d_1914 → d_1941) ───────────────────────────
path_verite = os.path.join(TEST_DIR, "verite_terrain_CA1.csv")
ca1[META_COLS + verite_day_cols].to_csv(path_verite, index=False)
print(f"✓ verite_terrain_CA1.csv       → {path_verite}")
print(f"  Shape : {ca1[META_COLS + verite_day_cols].shape}")

del ca1; gc.collect()

Jours historique   : 1885  (d_1 → d_1885)
Jours mois valid   : 28  (d_1886 → d_1913)
Jours vérité       : 28  (d_1914 → d_1941)

✓ historique_sales_CA1.csv     → C:\Users\ayady\Downloads\train\historique_sales_CA1.csv
  Shape : (3049, 1891)
✓ sales_du_mois_CA1.csv        → C:\Users\ayady\Downloads\test\sales_du_mois_CA1.csv
  Shape : (3049, 34)
✓ verite_terrain_CA1.csv       → C:\Users\ayady\Downloads\test\verite_terrain_CA1.csv
  Shape : (3049, 34)


0

## Étape 3 — Découpage des prix en 2 fichiers

In [4]:
# ── Lecture et filtrage prix CA_1 ─────────────────────────────────────────
print("Lecture sell_prices.csv ...")
prices_raw = pd.read_csv(PATH_PRICES)
prices_ca1 = prices_raw[prices_raw["store_id"] == STORE_ID].reset_index(drop=True)
del prices_raw
gc.collect()
print(f"  CA_1 prix : {len(prices_ca1)} lignes")

# ── Frontières semaines ───────────────────────────────────────────────────
cal_wk = pd.read_csv(PATH_CALENDAR, usecols=["d","wm_yr_wk"])
cal_wk["d_num"] = cal_wk["d"].str.extract(r"(\d+)")[0].astype(int)

# Semaines couvrant d_1 → d_1913 (historique)
wk_hist = cal_wk[cal_wk["d_num"] <= MOIS_END]["wm_yr_wk"].unique()
WK_HIST_MAX = int(max(wk_hist))

# Semaines couvrant d_1914 → d_1941 (période de prédiction uniquement)
wk_futur = cal_wk[
    (cal_wk["d_num"] >= MOIS_END + 1) & (cal_wk["d_num"] <= VERITE_END)
]["wm_yr_wk"].unique()
WK_FUTUR_MIN = int(min(wk_futur))
WK_FUTUR_MAX = int(max(wk_futur))

del cal_wk

print(f"WK_HIST_MAX  : {WK_HIST_MAX}  (contient d_{MOIS_END})")
print(f"WK_FUTUR_MIN : {WK_FUTUR_MIN} (contient d_{MOIS_END+1})")
print(f"WK_FUTUR_MAX : {WK_FUTUR_MAX} (contient d_{VERITE_END})")
print(f"Note : la semaine {WK_HIST_MAX} apparaît dans les deux fichiers "
      f"car elle chevauche d_{MOIS_END} et d_{MOIS_END+1} — c'est normal.")

# ── historique_prices : wk ≤ WK_HIST_MAX ───────────────────────────────────
prices_hist = prices_ca1[prices_ca1["wm_yr_wk"] <= WK_HIST_MAX].reset_index(drop=True)
path_hist_prices = os.path.join(TRAIN_DIR, "historique_prices_CA1.csv")
prices_hist.to_csv(path_hist_prices, index=False)
print(f"\n✓ historique_prices_CA1.csv")
print(f"  Shape    : {prices_hist.shape}")
print(f"  wm_yr_wk : {prices_hist['wm_yr_wk'].min()} → {prices_hist['wm_yr_wk'].max()}")

# ── prices_du_mois : WK_FUTUR_MIN ≤ wk ≤ WK_FUTUR_MAX (bornée strictement) ──
prices_futur = prices_ca1[
    (prices_ca1["wm_yr_wk"] >= WK_FUTUR_MIN) &
    (prices_ca1["wm_yr_wk"] <= WK_FUTUR_MAX)
].reset_index(drop=True)
path_futur_prices = os.path.join(TEST_DIR, "prices_du_mois_CA1.csv")
prices_futur.to_csv(path_futur_prices, index=False)
print(f"✓ prices_du_mois_CA1.csv")
print(f"  Shape    : {prices_futur.shape}")
print(f"  wm_yr_wk : {prices_futur['wm_yr_wk'].min()} → {prices_futur['wm_yr_wk'].max()}")

del prices_hist, prices_futur, prices_ca1
gc.collect()

Lecture sell_prices.csv ...
  CA_1 prix : 698412 lignes
WK_HIST_MAX  : 11613  (contient d_1913)
WK_FUTUR_MIN : 11613 (contient d_1914)
WK_FUTUR_MAX : 11617 (contient d_1941)
Note : la semaine 11613 apparaît dans les deux fichiers car elle chevauche d_1913 et d_1914 — c'est normal.

✓ historique_prices_CA1.csv
  Shape    : (674020, 4)
  wm_yr_wk : 11101 → 11613
✓ prices_du_mois_CA1.csv
  Shape    : (15245, 4)
  wm_yr_wk : 11613 → 11617


25

## Étape 4 — Copie du calendrier

In [5]:
# Copier le calendrier dans train/ pour que NB1 n'ait qu'un seul dossier à pointer
path_cal_dest = os.path.join(TRAIN_DIR, "calendar.csv")
shutil.copy(PATH_CALENDAR, path_cal_dest)
print(f"✓ calendar.csv                 → {path_cal_dest}")

✓ calendar.csv                 → C:\Users\ayady\Downloads\train\calendar.csv


## Étape 5 — Vérification : NB1 peut-il reconstituer train_df et prices_df ?

In [6]:
# ══════════════════════════════════════════════════════════════════════════
# DIAGNOSTIC COMPLET NB0
# ══════════════════════════════════════════════════════════════════════════
import pandas as pd
import os

print("="*70)
print("DIAGNOSTIC NB0 — Vérification des fichiers générés")
print("="*70)

# ── 1. Vérifier l'existence des 6 fichiers ─────────────────────────────────
files_check = [
    ("TRAIN", "historique_sales_CA1.csv",  TRAIN_DIR),
    ("TRAIN", "historique_prices_CA1.csv", TRAIN_DIR),
    ("TRAIN", "calendar.csv",              TRAIN_DIR),
    ("TEST",  "sales_du_mois_CA1.csv",     TEST_DIR),
    ("TEST",  "prices_du_mois_CA1.csv",    TEST_DIR),
    ("TEST",  "verite_terrain_CA1.csv",    TEST_DIR),
]

print("\n--- 1. Existence des fichiers ---")
for folder, fname, dirpath in files_check:
    path = os.path.join(dirpath, fname)
    exists = os.path.exists(path)
    size_mb = os.path.getsize(path)/1e6 if exists else 0
    print(f"  {'✓' if exists else '✗'} [{folder}] {fname:<28} {size_mb:>8.2f} Mo")

# ── 2. Recharger tous les fichiers ──────────────────────────────────────────
print("\n--- 2. Rechargement et vérification structure ---")
hist_sales  = pd.read_csv(os.path.join(TRAIN_DIR, "historique_sales_CA1.csv"))
mois_sales  = pd.read_csv(os.path.join(TEST_DIR,  "sales_du_mois_CA1.csv"))
verite      = pd.read_csv(os.path.join(TEST_DIR,  "verite_terrain_CA1.csv"))
hist_prices = pd.read_csv(os.path.join(TRAIN_DIR, "historique_prices_CA1.csv"))
futur_prices= pd.read_csv(os.path.join(TEST_DIR,  "prices_du_mois_CA1.csv"))
cal         = pd.read_csv(os.path.join(TRAIN_DIR, "calendar.csv"))

day_hist   = sorted([c for c in hist_sales.columns if c.startswith("d_")], key=lambda x:int(x[2:]))
day_mois   = sorted([c for c in mois_sales.columns if c.startswith("d_")], key=lambda x:int(x[2:]))
day_verite = sorted([c for c in verite.columns if c.startswith("d_")], key=lambda x:int(x[2:]))

print(f"  historique_sales : {hist_sales.shape}  | jours {day_hist[0]}→{day_hist[-1]} ({len(day_hist)})")
print(f"  sales_du_mois    : {mois_sales.shape}  | jours {day_mois[0]}→{day_mois[-1]} ({len(day_mois)})")
print(f"  verite_terrain   : {verite.shape}  | jours {day_verite[0]}→{day_verite[-1]} ({len(day_verite)})")
print(f"  historique_prices: {hist_prices.shape}  | wk {hist_prices['wm_yr_wk'].min()}→{hist_prices['wm_yr_wk'].max()}")
print(f"  prices_du_mois   : {futur_prices.shape}  | wk {futur_prices['wm_yr_wk'].min()}→{futur_prices['wm_yr_wk'].max()}")
print(f"  calendar         : {cal.shape}  | jours {cal['d'].iloc[0]}→{cal['d'].iloc[-1]}")

# ── 3. Vérifier continuité des jours (pas de trous) ─────────────────────────
print("\n--- 3. Continuité des jours ---")
days_full = [int(d[2:]) for d in day_hist] + [int(d[2:]) for d in day_mois] + [int(d[2:]) for d in day_verite]
days_full_sorted = sorted(days_full)
expected = list(range(days_full_sorted[0], days_full_sorted[-1]+1))
missing = sorted(set(expected) - set(days_full_sorted))
duplicate = len(days_full_sorted) != len(set(days_full_sorted))
print(f"  Range attendu : d_{expected[0]} → d_{expected[-1]} ({len(expected)} jours)")
print(f"  Jours manquants : {missing if missing else 'AUCUN ✓'}")
print(f"  Doublons        : {'OUI ✗' if duplicate else 'AUCUN ✓'}")

# ── 4. Vérifier cohérence produits (mêmes items partout) ────────────────────
print("\n--- 4. Cohérence des produits ---")
items_hist   = set(hist_sales["item_id"])
items_mois   = set(mois_sales["item_id"])
items_verite = set(verite["item_id"])
print(f"  Items historique : {len(items_hist)}")
print(f"  Items mois       : {len(items_mois)}")
print(f"  Items vérité     : {len(items_verite)}")
print(f"  Identiques partout : {'✓ OUI' if items_hist==items_mois==items_verite else '✗ NON — DIVERGENCE !'}")
if items_hist != items_mois:
    print(f"    Diff hist/mois : {items_hist.symmetric_difference(items_mois)}")
if items_hist != items_verite:
    print(f"    Diff hist/verite : {items_hist.symmetric_difference(items_verite)}")

# ── 5. Vérifier store_id unique ─────────────────────────────────────────────
print("\n--- 5. Store unique ---")
for name, df in [("hist_sales",hist_sales),("mois_sales",mois_sales),("verite",verite),
                  ("hist_prices",hist_prices),("futur_prices",futur_prices)]:
    stores = df["store_id"].unique()
    print(f"  {name:<14} store_id unique : {stores} {'✓' if len(stores)==1 else '✗ PROBLEME'}")

# ── 6. Vérifier chevauchement des semaines de prix (cohérence métier) ──────
print("\n--- 6. Chevauchement semaines prix ---")
wk_overlap = set(hist_prices["wm_yr_wk"]) & set(futur_prices["wm_yr_wk"])
print(f"  Semaines en commun hist/futur prices : {sorted(wk_overlap)}")
print(f"  (Normal si non vide — semaine chevauchant d_{MOIS_END}/d_{MOIS_END+1})")

# ── 7. Vérifier les prix couvrent bien tous les item_id × wm_yr_wk attendus ─
print("\n--- 7. Couverture des prix sur la période de prédiction ---")
cal_pred = cal[cal["d"].isin(day_verite)][["d","wm_yr_wk"]].drop_duplicates()
wk_pred_needed = set(cal_pred["wm_yr_wk"])
wk_pred_available = set(futur_prices["wm_yr_wk"])
wk_missing = wk_pred_needed - wk_pred_available
print(f"  Semaines nécessaires pour d_{day_verite[0]}-{day_verite[-1]} : {sorted(wk_pred_needed)}")
print(f"  Semaines disponibles dans prices_du_mois            : {sorted(wk_pred_available)}")
print(f"  Semaines manquantes : {sorted(wk_missing) if wk_missing else 'AUCUNE ✓'}")

# Vérifier couverture item × semaine (pas de prix manquant)
items_x_wk_needed = len(items_verite) * len(wk_pred_needed)
prices_avail_in_pred_window = futur_prices[
    futur_prices["wm_yr_wk"].isin(wk_pred_needed) & futur_prices["item_id"].isin(items_verite)
]
coverage_pct = len(prices_avail_in_pred_window) / items_x_wk_needed * 100 if items_x_wk_needed > 0 else 0
print(f"  Couverture prix (item×semaine) : {coverage_pct:.1f}% "
      f"({len(prices_avail_in_pred_window)}/{items_x_wk_needed})")

# ── 8. Stats globales ventes (sanity check) ─────────────────────────────────
print("\n--- 8. Sanity check ventes ---")
total_hist   = hist_sales[day_hist].sum().sum()
total_mois   = mois_sales[day_mois].sum().sum()
total_verite = verite[day_verite].sum().sum()
print(f"  Somme ventes historique (d_1-{MOIS_END})        : {total_hist:,.0f}")
print(f"  Somme ventes mois ({day_mois[0]}-{day_mois[-1]})         : {total_mois:,.0f}")
print(f"  Somme ventes vérité ({day_verite[0]}-{day_verite[-1]})       : {total_verite:,.0f}")
print(f"  Moyenne/jour historique : {total_hist/len(day_hist):.1f}")
print(f"  Moyenne/jour mois       : {total_mois/len(day_mois):.1f}")
print(f"  Moyenne/jour vérité     : {total_verite/len(day_verite):.1f}")

# ── 9. Bilan final ────────────────────────────────────────────────────────
print("\n" + "="*70)
all_ok = (not missing and not duplicate and items_hist==items_mois==items_verite
          and not wk_missing and coverage_pct > 95)
print(f"BILAN GLOBAL : {'✓ TOUT EST OK' if all_ok else '✗ PROBLEMES DETECTES — voir ci-dessus'}")
print("="*70)

del hist_sales, mois_sales, verite, hist_prices, futur_prices, cal, cal_pred

DIAGNOSTIC NB0 — Vérification des fichiers générés

--- 1. Existence des fichiers ---
  ✓ [TRAIN] historique_sales_CA1.csv        11.87 Mo
  ✓ [TRAIN] historique_prices_CA1.csv       20.70 Mo
  ✓ [TRAIN] calendar.csv                     0.10 Mo
  ✓ [TEST] sales_du_mois_CA1.csv            0.39 Mo
  ✓ [TEST] prices_du_mois_CA1.csv           0.47 Mo
  ✓ [TEST] verite_terrain_CA1.csv           0.39 Mo

--- 2. Rechargement et vérification structure ---
  historique_sales : (3049, 1891)  | jours d_1→d_1885 (1885)
  sales_du_mois    : (3049, 34)  | jours d_1886→d_1913 (28)
  verite_terrain   : (3049, 34)  | jours d_1914→d_1941 (28)
  historique_prices: (674020, 4)  | wk 11101→11613
  prices_du_mois   : (15245, 4)  | wk 11613→11617
  calendar         : (1969, 14)  | jours d_1→d_1969

--- 3. Continuité des jours ---
  Range attendu : d_1 → d_1941 (1941 jours)
  Jours manquants : AUCUN ✓
  Doublons        : AUCUN ✓

--- 4. Cohérence des produits ---
  Items historique : 3049
  Items mois       :